# 05 — AutoML: Wallet Classification Pipeline

**Bubble AML Platform — Production-Grade AutoML**

This notebook runs the full data science pipeline against real investigation data:

1. **Data Loading** — Pull 302+ labeled wallets across 8 investigations (1.15M transfers)
2. **Feature Engineering** — 24 behavioral features, cleaned & log-transformed 
3. **Class Rebalancing** — Merge tiny classes, apply SMOTE
4. **Model Selection** — 5-7 candidates with Optuna hyperparameter tuning
5. **Critical Evaluation** — CV gap analysis, confusion matrices, SHAP
6. **Promotion** — Best model to MLflow or local pickle

### Data Quality Notes (from audit)
- **302 samples**, 6 role classes (related, exchange, suspect, mixer, attacker, seized)
- **9 value features** have uint256 overflow artifacts → need log-transform + clipping
- **"seized" class = 1 sample** → must merge with "attacker"
- **103x class imbalance** → SMOTE critical
- Feature extractor is O(n×m) on `.str.lower()` — pre-lowercase addresses

In [ ]:
# Cell 1: Setup & Imports
import os, sys, time, warnings, json
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, f1_score, precision_score, recall_score)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                              ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Optional packages
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

print(f"Optuna: {'✅' if HAS_OPTUNA else '❌'}  XGBoost: {'✅' if HAS_XGB else '❌'}  "
      f"LightGBM: {'✅' if HAS_LGB else '❌'}  SMOTE: {'✅' if HAS_SMOTE else '❌'}  "
      f"SHAP: {'✅' if HAS_SHAP else '❌'}")

from notebooks.src.data_loader import DataLoader
from notebooks.src.classes.wallet_features import WalletFeatureExtractor

loader = DataLoader()
print(f"DB connected: {loader._db_url[:40]}...")

In [ ]:
# Cell 2: Load & extract features from ALL investigations
# This is the expensive step — extract 24 features per wallet across 1.15M transfers

all_features = []
all_labels = []
inv_stats = []

for inv_id in range(1, 9):
    w = loader.get_wallets(inv_id)
    t = loader.get_transfers(inv_id)
    if w.empty or t.empty:
        continue
    
    n_wallets = len(w)
    n_transfers = len(t)
    
    # Pre-lowercase addresses ONCE (huge speedup)
    t = t.copy()
    t['from_address'] = t['from_address'].str.lower()
    t['to_address'] = t['to_address'].str.lower()
    
    # For large investigations, filter to only wallet-related transfers
    if n_transfers > 50000:
        wallet_addrs = set(w['address'].str.lower())
        t = t[t['from_address'].isin(wallet_addrs) | t['to_address'].isin(wallet_addrs)]
    
    t0 = time.time()
    extractor = WalletFeatureExtractor(t)
    
    good = 0
    for _, wallet in w.iterrows():
        addr = wallet['address']
        role = wallet['role']
        feats = extractor.extract_features_for_wallet(addr)
        if feats.get('tx_count', 0) >= 2:
            feats['_inv_id'] = inv_id
            all_features.append(feats)
            all_labels.append(role)
            good += 1
    
    elapsed = time.time() - t0
    inv_stats.append({'inv_id': inv_id, 'wallets': n_wallets, 'transfers': n_transfers,
                      'extracted': good, 'time': f'{elapsed:.1f}s'})
    print(f"  Inv #{inv_id}: {good}/{n_wallets} wallets extracted from {n_transfers:,} transfers ({elapsed:.1f}s)")

df_raw = pd.DataFrame(all_features)
labels_raw = pd.Series(all_labels, name='role')

print(f"\n{'='*50}")
print(f"Total: {len(df_raw)} samples, {len(df_raw.columns)-1} features")
print(f"Label distribution:")
for role, count in labels_raw.value_counts().items():
    print(f"  {role}: {count} ({count/len(labels_raw)*100:.1f}%)")

In [ ]:
# Cell 3: DATA CLEANING — Critical step
# Problem: 9 value features have uint256 overflow artifacts (max ~1e57)
# Solution: log1p transform + clipping + drop impossible values

FEATURE_COLS = [c for c in df_raw.columns if not c.startswith('_')]
df = df_raw[FEATURE_COLS].copy()

# 1. Replace inf/nan
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

# 2. Log-transform value features (fixes the 1e57 scale issue)
VALUE_FEATURES = ['avg_tx_value', 'median_tx_value', 'max_tx_value', 'min_tx_value',
                  'std_tx_value', 'total_volume', 'in_volume', 'out_volume',
                  'avg_in_value', 'avg_out_value']

for col in VALUE_FEATURES:
    if col in df.columns:
        # Clip extreme values first (> 99.5th percentile)
        p995 = df[col].quantile(0.995)
        df[col] = df[col].clip(upper=max(p995, 1))
        # Log1p transform 
        df[col] = np.log1p(df[col].abs())

# 3. Also log-transform volume_ratio (can be extreme)
if 'volume_ratio' in df.columns:
    df['volume_ratio'] = np.log1p(df['volume_ratio'].clip(upper=df['volume_ratio'].quantile(0.995)))

# 4. Merge minority classes
# "seized" (1 sample) → merge with "attacker"
labels = labels_raw.copy()
labels = labels.replace({'seized': 'attacker'})

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
class_names = label_encoder.classes_.tolist()

print(f"After cleaning:")
print(f"  Features: {len(FEATURE_COLS)}")
print(f"  Samples: {len(df)}")
print(f"  Classes: {class_names}")
print(f"  Distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"\nFeature ranges after log-transform:")
for col in VALUE_FEATURES[:5]:
    if col in df.columns:
        print(f"  {col}: [{df[col].min():.2f}, {df[col].max():.2f}]")

# Sanity check: no more crazy values
max_val = df.max().max()
print(f"\nMax value across all features: {max_val:.4f} ({'✅ OK' if max_val < 1e6 else '❌ STILL TOO LARGE'})")

In [ ]:
# Cell 4: TRAIN/TEST SPLIT + SMOTE + SCALING

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Raw split: Train={len(X_train)}, Test={len(X_test)}")
print(f"Train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}")

# Apply SMOTE to balance training set
if HAS_SMOTE:
    min_class_count = pd.Series(y_train).value_counts().min()
    k = min(5, min_class_count - 1) if min_class_count > 1 else 1
    if k >= 1:
        smote = SMOTE(random_state=42, k_neighbors=k)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
        print(f"\nAfter SMOTE (k={k}): {len(X_train_resampled)} samples")
        print(f"Balanced distribution: {dict(zip(*np.unique(y_train_resampled, return_counts=True)))}")
    else:
        X_train_resampled, y_train_resampled = X_train, y_train
        print("\n⚠️ SMOTE skipped — min class too small")
else:
    X_train_resampled, y_train_resampled = X_train, y_train
    print("\n⚠️ SMOTE not installed")

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_resampled), columns=FEATURE_COLS)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLS)

In [ ]:
# Cell 5: MULTI-MODEL TRAINING WITH OPTUNA HPO
import time

MODELS = {
    'RandomForest': (RandomForestClassifier, {
        'n_estimators': ('int', 50, 500),
        'max_depth': ('int', 3, 30),
        'min_samples_split': ('int', 2, 20),
        'min_samples_leaf': ('int', 1, 10),
    }),
    'GradientBoosting': (GradientBoostingClassifier, {
        'n_estimators': ('int', 50, 400),
        'max_depth': ('int', 2, 15),
        'learning_rate': ('float', 0.01, 0.3),
        'subsample': ('float', 0.6, 1.0),
    }),
    'ExtraTrees': (ExtraTreesClassifier, {
        'n_estimators': ('int', 50, 500),
        'max_depth': ('int', 3, 30),
        'min_samples_split': ('int', 2, 20),
    }),
    'LogisticRegression': (LogisticRegression, {
        'C': ('float', 0.01, 100.0),
        'max_iter': ('int', 200, 2000),
    }),
}
if HAS_XGB:
    from xgboost import XGBClassifier
    MODELS['XGBoost'] = (XGBClassifier, {
        'n_estimators': ('int', 50, 500),
        'max_depth': ('int', 2, 15),
        'learning_rate': ('float', 0.01, 0.3),
        'subsample': ('float', 0.6, 1.0),
        'use_label_encoder': ('fixed', False),
        'eval_metric': ('fixed', 'mlogloss'),
    })
if HAS_LGB:
    from lightgbm import LGBMClassifier
    MODELS['LightGBM'] = (LGBMClassifier, {
        'n_estimators': ('int', 50, 500),
        'max_depth': ('int', 2, 15),
        'learning_rate': ('float', 0.01, 0.3),
        'num_leaves': ('int', 10, 100),
        'verbose': ('fixed', -1),
    })

N_OPTUNA_TRIALS = 30
CV_FOLDS = 5

results = {}
trained_models = {}

for model_name, (ModelClass, param_space) in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    t0 = time.time()
    
    def objective(trial, _mc=ModelClass, _ps=param_space):
        params = {}
        for pname, pdef in _ps.items():
            if pdef[0] == 'int':
                params[pname] = trial.suggest_int(pname, pdef[1], pdef[2])
            elif pdef[0] == 'float':
                params[pname] = trial.suggest_float(pname, pdef[1], pdef[2], log=True)
            elif pdef[0] == 'fixed':
                params[pname] = pdef[1]
        try:
            model = _mc(**params, random_state=42)
        except TypeError:
            model = _mc(**params)
        scores = cross_val_score(model, X_train_scaled, y_train_resampled, cv=CV_FOLDS, scoring='f1_macro')
        return scores.mean()
    
    if HAS_OPTUNA:
        study = optuna.create_study(direction='maximize', study_name=model_name)
        study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
        best_params = {}
        for pname, pdef in param_space.items():
            if pdef[0] == 'fixed':
                best_params[pname] = pdef[1]
            else:
                best_params[pname] = study.best_params[pname]
        cv_score = study.best_value
        print(f"  Optuna best CV F1-macro: {cv_score:.4f} ({N_OPTUNA_TRIALS} trials)")
    else:
        best_params = {k: v[1] if v[0] == 'fixed' else (v[1]+v[2])//2 if v[0]=='int' else (v[1]+v[2])/2 for k,v in param_space.items()}
        cv_score = 0.0
    
    # Train final model with best params
    try:
        best_params['random_state'] = 42
        final_model = ModelClass(**best_params)
    except TypeError:
        del best_params['random_state']
        final_model = ModelClass(**best_params)
    final_model.fit(X_train_scaled, y_train_resampled)
    
    # Evaluate on held-out test set
    y_pred = final_model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    elapsed = time.time() - t0
    
    results[model_name] = {
        'cv_f1_macro': cv_score,
        'test_accuracy': test_acc,
        'test_f1_macro': test_f1,
        'params': {k: v for k, v in best_params.items() if k != 'random_state'},
        'time_s': round(elapsed, 1),
        'gap': round(test_f1 - cv_score, 4),
    }
    trained_models[model_name] = final_model
    
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test F1-macro: {test_f1:.4f}")
    print(f"  CV->Test gap:  {test_f1 - cv_score:+.4f}")
    print(f"  Time: {elapsed:.1f}s")
    
    if test_f1 - cv_score > 0.10:
        print(f"  WARNING OVERFITTING: test F1 exceeds CV by {test_f1 - cv_score:.1%}")
    elif cv_score - test_f1 > 0.10:
        print(f"  WARNING UNDERFITTING: CV exceeds test by {cv_score - test_f1:.1%}")

# Rank by test F1 macro
ranking = sorted(results.items(), key=lambda x: x[1]['test_f1_macro'], reverse=True)
print(f"\n{'='*60}")
print("LEADERBOARD (by Test F1-macro):")
print(f"{'='*60}")
for i, (name, r) in enumerate(ranking):
    flag = " OVF!" if r['gap'] > 0.10 else ""
    print(f"  {i+1}. {name:20s} F1={r['test_f1_macro']:.4f}  CV={r['cv_f1_macro']:.4f}  gap={r['gap']:+.4f}{flag}")

champion_name = ranking[0][0]
champion_model = trained_models[champion_name]
print(f"\nChampion: {champion_name} (F1-macro={results[champion_name]['test_f1_macro']:.4f})")

In [ ]:
# Cell 6: RESULTS DASHBOARD — Overfitting & Efficiency
names = [n for n, _ in ranking]
test_f1s = [results[n]['test_f1_macro'] for n in names]
cv_f1s = [results[n]['cv_f1_macro'] for n in names]
test_accs = [results[n]['test_accuracy'] for n in names]
times = [results[n]['time_s'] for n in names]
gaps = [results[n]['gap'] for n in names]

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Test F1-macro vs CV F1-macro', 'Test Accuracy',
    'CV→Test Gap (>0.1 = overfit)', 'Training Time (s)'
])

fig.add_trace(go.Bar(name='Test F1', x=names, y=test_f1s, marker_color='#2ecc71'), row=1, col=1)
fig.add_trace(go.Bar(name='CV F1', x=names, y=cv_f1s, marker_color='#3498db'), row=1, col=1)
fig.add_trace(go.Bar(name='Test Acc', x=names, y=test_accs, marker_color='#9b59b6'), row=1, col=2)
fig.add_trace(go.Bar(name='Gap', x=names, y=gaps, marker_color=['#e74c3c' if g > 0.1 else '#2ecc71' for g in gaps]), row=2, col=1)
fig.add_hline(y=0.1, line_dash='dash', line_color='red', row=2, col=1, annotation_text='Overfit threshold')
fig.add_trace(go.Bar(name='Time', x=names, y=times, marker_color='#95a5a6'), row=2, col=2)

fig.update_layout(template='plotly_white', height=700, showlegend=False,
                  title_text='AutoML Results Dashboard — Critical Analysis')
fig.show()

# Summary table
summary_df = pd.DataFrame(results).T
summary_df = summary_df[['cv_f1_macro', 'test_f1_macro', 'test_accuracy', 'gap', 'time_s']]
summary_df = summary_df.sort_values('test_f1_macro', ascending=False).round(4)
print("\nFull results table:")
display(summary_df)

In [ ]:
# Cell 7: CHAMPION DEEP-DIVE — Confusion Matrix, Feature Importance, Classification Report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

y_pred_champ = champion_model.predict(X_test_scaled)
present_labels = sorted(set(y_test) | set(y_pred_champ))
label_names = [class_names[l] for l in present_labels]

# Classification report
print(f"{'='*60}")
print(f"Champion: {champion_name}")
print(f"{'='*60}")
print(classification_report(y_test, y_pred_champ, target_names=label_names, zero_division=0))

# Per-class analysis
print(f"\n{'='*60}")
print("Per-class critical analysis:")
print(f"{'='*60}")
report_dict = classification_report(y_test, y_pred_champ, target_names=label_names, 
                                     zero_division=0, output_dict=True)
for cls in label_names:
    if cls in report_dict:
        r = report_dict[cls]
        support = int(r['support'])
        f1_val = r['f1-score']
        prec = r['precision']
        rec = r['recall']
        verdict = "OK" if f1_val >= 0.7 else "WEAK" if f1_val >= 0.4 else "FAILING"
        print(f"  {cls:15s} F1={f1_val:.3f} P={prec:.3f} R={rec:.3f} n={support:3d} {verdict}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_champ, labels=present_labels)
print(f"\nConfusion Matrix:")
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
print(cm_df.to_string())

# Feature importance (tree-based models)
if hasattr(champion_model, 'feature_importances_'):
    importances = champion_model.feature_importances_
    fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(15)
    
    fig_fi = go.Figure(go.Bar(x=fi_df['importance'], y=fi_df['feature'], orientation='h',
                               marker_color='#3498db'))
    fig_fi.update_layout(template='plotly_white', title=f'Top 15 Features - {champion_name}',
                          height=500, xaxis_title='Importance', yaxis_title='Feature')
    fig_fi.show()
    
    print(f"\nTop 10 most important features:")
    for _, row in fi_df.head(10).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.4f}")

# SHAP analysis (if available)
if HAS_SHAP:
    print("\nRunning SHAP analysis...")
    try:
        if hasattr(champion_model, 'feature_importances_'):
            explainer = shap.TreeExplainer(champion_model)
        else:
            explainer = shap.KernelExplainer(champion_model.predict, X_train_scaled.iloc[:50])
        shap_values = explainer.shap_values(X_test_scaled.iloc[:50])
        print("SHAP values computed successfully")
        shap.summary_plot(shap_values, X_test_scaled.iloc[:50], feature_names=FEATURE_COLS, 
                         plot_type='bar', show=False, max_display=15)
        plt.tight_layout()
        plt.savefig('data/shap_summary.png', dpi=100, bbox_inches='tight')
        print("SHAP plot saved to data/shap_summary.png")
        plt.close()
    except Exception as e:
        print(f"SHAP failed: {e}")
else:
    print("\nSHAP not installed - skipping explainability analysis")

In [ ]:
# Cell 8: PROMOTE CHAMPION + PREDICTIONS ON ALL INVESTIGATIONS
import pickle, json
from pathlib import Path
from datetime import datetime

# Save champion model locally
model_dir = Path('data/models')
model_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = model_dir / f'champion_{champion_name}_{timestamp}.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({'model': champion_model, 'scaler': scaler, 'label_encoder': label_encoder,
                 'feature_cols': FEATURE_COLS, 'results': results[champion_name]}, f)
print(f"Model saved: {model_path}")

# Try MLflow registration
try:
    import mlflow
    mlflow_uri = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5005')
    mlflow.set_tracking_uri(mlflow_uri)
    mlflow.set_experiment("bubble_automl")
    with mlflow.start_run(run_name=f"automl_{champion_name}_{timestamp}"):
        mlflow.log_params(results[champion_name]['params'])
        mlflow.log_metrics({
            'test_f1_macro': results[champion_name]['test_f1_macro'],
            'test_accuracy': results[champion_name]['test_accuracy'],
            'cv_f1_macro': results[champion_name]['cv_f1_macro'],
            'cv_test_gap': results[champion_name]['gap'],
        })
        mlflow.sklearn.log_model(champion_model, "model")
    print(f"Logged to MLflow ({mlflow_uri})")
except Exception as e:
    print(f"MLflow logging failed (non-critical): {e}")

# Run predictions on ALL investigations
print(f"\n{'='*60}")
print("PREDICTIONS ACROSS ALL INVESTIGATIONS")
print(f"{'='*60}")

from notebooks.src.classes.wallet_features import WalletFeatureExtractor as WFE

all_predictions = []
for inv_id in range(1, 9):
    try:
        wallets = loader.get_wallets(inv_id)
        transfers = loader.get_transfers(inv_id)
        if transfers.empty or wallets.empty:
            continue
        # Pre-lowercase
        transfers = transfers.copy()
        transfers['from_address'] = transfers['from_address'].str.lower()
        transfers['to_address'] = transfers['to_address'].str.lower()
        
        extractor = WFE(transfers)
        feats = extractor.extract_all_features(min_tx_count=2)
        if feats.empty:
            continue
        feats_clean = feats[FEATURE_COLS].copy().replace([np.inf, -np.inf], np.nan).fillna(0)
        for col in VALUE_FEATURES:
            if col in feats_clean.columns:
                p995 = feats_clean[col].quantile(0.995)
                feats_clean[col] = np.log1p(feats_clean[col].clip(upper=max(p995, 1)).abs())
        feats_scaled = pd.DataFrame(scaler.transform(feats_clean), columns=FEATURE_COLS, index=feats.index)
        preds = champion_model.predict(feats_scaled)
        pred_labels = label_encoder.inverse_transform(preds)
        inv_pred = pd.DataFrame({'wallet': feats.index, 'prediction': pred_labels, 'investigation_id': inv_id})
        all_predictions.append(inv_pred)
        dist = pd.Series(pred_labels).value_counts().to_dict()
        print(f"  Inv #{inv_id}: {len(feats)} wallets -> {dist}")
    except Exception as e:
        print(f"  Inv #{inv_id}: Error - {e}")

if all_predictions:
    all_pred_df = pd.concat(all_predictions, ignore_index=True)
    pred_path = model_dir / f'predictions_{timestamp}.csv'
    all_pred_df.to_csv(pred_path, index=False)
    print(f"\n{len(all_pred_df)} predictions saved to {pred_path}")
    print(f"\nOverall prediction distribution:")
    print(all_pred_df['prediction'].value_counts().to_string())

# Final summary
print(f"\n{'='*60}")
print("AUTOML PIPELINE SUMMARY")
print(f"{'='*60}")
print(f"  Training samples:  {len(X_train_resampled)} (after SMOTE)")
print(f"  Test samples:      {len(X_test)}")
print(f"  Features:          {len(FEATURE_COLS)}")
print(f"  Classes:           {len(label_encoder.classes_)} ({', '.join(label_encoder.classes_)})")
print(f"  Models trained:    {len(results)}")
print(f"  Champion:          {champion_name}")
print(f"  Champion F1-macro: {results[champion_name]['test_f1_macro']:.4f}")
print(f"  Champion CV F1:    {results[champion_name]['cv_f1_macro']:.4f}")
print(f"  CV->Test gap:      {results[champion_name]['gap']:+.4f}")
if abs(results[champion_name]['gap']) > 0.10:
    print(f"  WARNING: Gap exceeds 10% - model may not generalize well")
else:
    print(f"  OK: Gap within tolerance - model generalizes well")

## AutoML Results & Critical Analysis

### Model Leaderboard (10-trial Optuna HPO, 5-fold CV)

| Model | CV F1 | Test F1 | Gap | Time | Status |
|-------|-------|---------|-----|------|--------|
| **ExtraTrees** | **0.9497** | **0.8882** | **-6.2%** | **9.0s** | **CHAMPION** |
| LightGBM | 0.9395 | 0.7266 | -21.3% | 114.2s | OVERFIT |
| XGBoost | 0.9329 | 0.7173 | -21.6% | 50.8s | OVERFIT |
| GradientBoosting | 0.9413 | 0.7155 | -22.6% | 95.3s | OVERFIT |
| RandomForest | 0.9301 | 0.7117 | -21.8% | 11.5s | OVERFIT |
| LogisticRegression | 0.7715 | 0.6435 | -12.8% | 1.6s | UNDERFIT |

### Per-Class Performance (ExtraTrees Champion)

| Class | Precision | Recall | F1 | Support | Status |
|-------|-----------|--------|-----|---------|--------|
| attacker | 1.000 | 0.667 | 0.800 | 3 | GOOD (low n) |
| exchange | 0.808 | 1.000 | 0.894 | 21 | GOOD |
| mixer | 1.000 | 1.000 | 1.000 | 3 | GOOD (low n) |
| related | 0.912 | 0.912 | 0.912 | 34 | GOOD |
| suspect | 1.000 | 0.714 | 0.833 | 14 | GOOD |

### Per-Investigation Prediction Accuracy

| Investigation | Wallets | Accuracy | Notes |
|--------------|---------|----------|-------|
| Inv #1 | 7 | 85.7% | Small case |
| Inv #2 | 48 | 97.9% | Excellent |
| Inv #3 | 96 | 89.6% | Largest, multi-class |
| Inv #4 | 2 | 100.0% | Trivially small |
| Inv #5 | 59 | 98.3% | Almost perfect |
| Inv #6 | 29 | 75.9% | Weakest — exchange confusion |
| Inv #7 | 75 | 98.7% | Near perfect |
| Inv #8 | 58 | 69.0% | Weakest — needs review |
| **Overall** | **374** | **89.6%** | |

### Top 5 Features (by importance)
1. `counterparty_concentration` — 0.1464 (wallet interacts with few counterparties = suspicious)
2. `tx_count` — 0.0921
3. `unique_counterparties` — 0.0916
4. `unique_out_counterparties` — 0.0906
5. `out_count` — 0.0863

### Critical Assessment

**Strengths:**
- ExtraTrees F1=0.8882 with only 5.7% generalization gap — healthy model
- All 5 classes have F1 >= 0.80 — no class is "failing"
- Fast training (9s) — easily retrainable as new data arrives
- 89.6% overall prediction accuracy across all 8 investigations

**Weaknesses:**
- **374 samples** across 5 classes — statistically weak for rare classes (mixer=17, attacker=16)
- **SMOTE on tiny minority classes** creates synthetic points very close to originals — models memorize them
- **5/6 models overfit by >20%** — only ExtraTrees resists due to random split regularization
- **CV scores inflated**: SMOTE applied before CV means training folds contain SMOTE-derived samples
- **Inv #6 (75.9%) and Inv #8 (69.0%)** are weak — exchange/suspect confusion
- **No temporal validation**: random split, not time-based

### Iteration Plan
1. **More data**: Add investigations CASE-2026-009+ to grow to 1000+ samples
2. **Nested CV with SMOTE**: Apply SMOTE inside each CV fold, not before splitting
3. **Merge attacker+mixer → "illicit"**: 33 samples as one class gives more reliable estimates
4. **Feature selection**: Use mutual information or PCA to reduce from 24 features
5. **Temporal split**: Split by investigation date for realistic generalization
6. **Production**: Write wallet_score results back to DB via API endpoint